# Step 08 — LLM on the validation set

**Input** — `data/validation/06_validation_set.csv`

**Output** — `llm_predictions.parquet` + `llm_comparison.csv`

The `sentence` column is already the rendered prompt, built by notebook 06 from

the frozen buildings file in exactly the format `llm_utils.row_to_llm_input`

produces. Every row is sent to the model as-is — nothing is re-rendered, so the

model sees precisely the evidence the human annotator saw.

Every row is classified **fresh, under the current `llm_utils.SYSTEM_PROMPT`**.

Predictions from an earlier prompt version are never mixed in: doing so would put

two different classifiers behind one accuracy number.

`LLM_MAX_WORKERS = 1` (config) — one request at a time, deliberately. 889 rows at

~7 s/row is roughly 1.5–2 hours. The run checkpoints every `LLM_CHUNK_SIZE` rows,

so re-running this notebook after an interruption resumes where it stopped

instead of re-paying for completed rows.

> **Read this before quoting the score.** The `Predicted_activities` /

> `Bosserhof_class_predicted` columns ARE an earlier run of this same model. On

> every row the validator marked *green*, prediction and truth are equal **by

> construction**, so a green-row accuracy is self-referential. What a fresh run

> genuinely measures is whether the model still agrees with itself — see

> notebook 10.

In [1]:
import sys

sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))

import time

import pandas as pd

from config import (VALIDATION_SET_FILE, arm_predictions, arm_comparison,

                    LLM_CHUNK_SIZE, LLM_MODEL, LLM_REASONING,

                    LLM_RETRY_SWEEPS, LLM_SWEEP_PAUSE_SEC)

from llm_utils import predict_row, normalise_mid_labels, SYSTEM_PROMPT

from validation_utils import (decode_final_validation_set, collapse_to_zone_activities,

                              resolve_prediction_bosserhof, build_comparison)

ARM = 'llm'

CHECKPOINT = arm_predictions(ARM)

pd.set_option('display.width', 200)

print(f'model={LLM_MODEL}  reasoning={LLM_REASONING}  prompt={len(SYSTEM_PROMPT):,} chars')

model=gpt-oss-120b  reasoning=high  prompt=11,982 chars


## 1. Load the validation set and decode its ground truth

In [2]:
val = decode_final_validation_set(pd.read_csv(VALIDATION_SET_FILE))

val['gml_id'] = val['gml_id'].astype(str)

print(f'{len(val):,} validated buildings')

print(f"  activities scoreable : {val['activities_truth'].notna().sum():,}")

print(f"  bosserhof scoreable  : {val['bosserhof_truth'].notna().sum():,}")

889 validated buildings
  activities scoreable : 876
  bosserhof scoreable  : 877


## 2. Classify — resumable, and it will not give up on a row

**A row that errored is not "done".** The checkpoint stores failures alongside

successes so they can be inspected, but `still_missing()` counts only rows with a

real answer, so every resume — and every retry sweep — picks the failures back up.

Getting this wrong is silent: an errored row would sit in the checkpoint forever,

be treated as complete, and score as an empty prediction, which looks like the

model answering "nothing" rather than the model never being asked.

After the main pass, up to `LLM_RETRY_SWEEPS` further passes re-attempt whatever is

still missing, pausing longer each time — the KI-Toolbox rate limit is undocumented

and rejects requests made in quick succession, so the fix is to come back later,

not to hammer it. The cell ends by asserting nothing is left unanswered, so a

partial run cannot quietly become a published score.

Delete `data/validation/llm_predictions.parquet` to force a completely fresh run —

which you must do after changing the system prompt.

In [3]:
def append_checkpoint(path, rows):

    # keep='last' is what lets a successful retry overwrite an earlier failure.

    new = pd.DataFrame(rows)

    if path.exists():

        prior = pd.read_parquet(path)

        prior['gml_id'] = prior['gml_id'].astype(str)

        new = pd.concat([prior, new], ignore_index=True).drop_duplicates('gml_id', keep='last')

    path.parent.mkdir(parents=True, exist_ok=True)

    new.to_parquet(path, index=False)

def still_missing():

    """Validation rows with no SUCCESSFUL prediction yet — errors count as missing."""

    if not CHECKPOINT.exists():

        return val

    prior = pd.read_parquet(CHECKPOINT)

    prior['gml_id'] = prior['gml_id'].astype(str)

    answered = set(prior.loc[prior['error'].isna(), 'gml_id'])

    return val[~val['gml_id'].isin(answered)]

def classify(rows, label, abort_on_bad_first_chunk=False):

    chunks = [rows.iloc[i:i + LLM_CHUNK_SIZE] for i in range(0, len(rows), LLM_CHUNK_SIZE)]

    t0, n_called = time.time(), 0

    for c, chunk in enumerate(chunks, 1):

        results = [predict_row(r.gml_id, r.sentence, fields='full',

                               src_file=VALIDATION_SET_FILE.name)

                   for r in chunk.itertuples(index=False)]

        n_err = sum(1 for r in results if r['error'] is not None)

        append_checkpoint(CHECKPOINT, results)

        n_called += len(results)

        rate = (time.time() - t0) / n_called

        print(f'{label} chunk {c}/{len(chunks)} | {n_called:,}/{len(rows):,} | '

              f'errors {n_err} | {rate:.1f} s/row | '

              f'ETA {(len(rows) - n_called) * rate / 60:.1f} min')

        # Only on the first pass: a bad token or a moved endpoint fails every row,

        # and no amount of sweeping fixes it. Fail in 25 calls, not in 889.

        if abort_on_bad_first_chunk and c == 1 and n_err > 0.2 * len(chunk):

            for r in [r for r in results if r['error']][:3]:

                print('   ', r['error'])

            raise RuntimeError(

                f'{n_err}/{len(chunk)} failed in the FIRST chunk — aborting rather than '

                'burning hours on a systematic fault. Check the token and the endpoint. '

                'Completed rows are checkpointed, so a re-run resumes.')

todo = still_missing()

print(f'already answered : {len(val) - len(todo):,} / {len(val):,}')

print(f'to classify      : {len(todo):,}')

if len(todo):

    classify(todo, 'pass 1', abort_on_bad_first_chunk=True)

# --- retry sweeps: never leave a row unanswered -----------------------------

for sweep in range(1, LLM_RETRY_SWEEPS + 1):

    failed = still_missing()

    if failed.empty:

        break

    pause = LLM_SWEEP_PAUSE_SEC * sweep

    print(f'\n{len(failed):,} row(s) still unanswered — sweep {sweep}/{LLM_RETRY_SWEEPS} '

          f'after a {pause}s pause')

    time.sleep(pause)

    classify(failed, f'sweep {sweep}')

outstanding = still_missing()

assert outstanding.empty, (

    f'{len(outstanding)} row(s) still have no prediction after {LLM_RETRY_SWEEPS} sweeps: '

    f'{sorted(outstanding["gml_id"])[:10]}. Re-run this cell to sweep again — progress is '

    'checkpointed, so it costs only the failures. Refusing to score a partial run.')

print(f'\nall {len(val):,} rows answered.')

already answered : 889 / 889
to classify      : 0

all 889 rows answered.


In [4]:
preds = pd.read_parquet(CHECKPOINT)

preds['gml_id'] = preds['gml_id'].astype(str)

preds = preds[preds['gml_id'].isin(val['gml_id'])].drop_duplicates('gml_id', keep='last')

missing = set(val['gml_id']) - set(preds['gml_id'])

assert not missing, f'{len(missing)} rows were never classified: {sorted(missing)[:5]}'

assert preds['error'].isna().all(), (

    f"{preds['error'].notna().sum()} rows still carry an error — the retry sweeps above "

    'should have made this unreachable')

preds['mid_labels'] = preds['mid_labels'].map(normalise_mid_labels)

preds['pred_zone_activities'] = preds['mid_labels'].map(collapse_to_zone_activities)

preds['pred_bosserhof'] = preds['bosserhof_class'].map(resolve_prediction_bosserhof)

print(f'{len(preds):,} predictions ready, none failed')

889 predictions ready, none failed


## 3. Side-by-side review sheet

In [5]:
comparison = build_comparison(val, preds)

comparison.to_csv(arm_comparison(ARM), index=False, encoding='utf-8')

print(f'wrote {arm_comparison(ARM).name}  ({len(comparison):,} rows)')

comparison.drop(columns='sentence').head(8)

wrote llm_comparison.csv  (889 rows)


,gml_id,source_gml_id,osm_names,volume_m3,act_prefilled,act_human_verdict,act_truth,act_predicted,act_match,act_extra,act_missing,boss_prefilled,boss_human_verdict,boss_truth,boss_predicted,boss_match
0,236928,UUID_968d084c-1df3-4297-8086-87e0bc8ec21a,['Enoteca Vetrone'],1554.252341,"['Leisure', 'Workers']",green,Leisure; Workers,Leisure; Workers,1,,,restaurants gastronomy,green,restaurants gastronomy,restaurants gastronomy,1
1,388629,DENILD2703375395269990650_1,"[""Deutsche Bank;Ernsting's family""]",17437.047953,"['Retail_Daily', 'Workers', 'Retail_Non-Daily']",red,Retail_Non-Daily; Workers,Retail_Daily; Retail_Non-Daily; Workers,0,Retail_Daily,,retail small scale,green,retail small scale,retail small scale,1
2,81391,DENILD0300007qJ5,NaN,726.737078,['Workers'],green,Workers,(none),0,,Workers,industrial operations production,green,industrial operations production,(no class),0
3,36228,DENILD0100005y5m,"['Nazar Trockenfrüchte', ""Sara's Collection"", ...",12947.777792,"['Retail_Daily', 'Retail_Non-Daily', 'Universi...",red,Retail_Daily; Retail_Non-Daily; University; Wo...,Leisure; Retail_Daily; Retail_Non-Daily; Unive...,0,Leisure,,retail small scale,green,retail small scale,retail small scale,1
4,556581,DENILD12706506665755496596_2,['Reni'],1371.218910,"['Leisure', 'Workers']",green,Leisure; Workers,Leisure; Workers,1,,,hotels,green,hotels,hotels,1
5,389444,DENILD313391039270178853_10,NaN,105.044662,"['Workers', 'Retail_Daily']",red,Retail_Daily; Retail_Non-Daily; Workers,Retail_Daily; Workers,0,,Retail_Non-Daily,customer oriented services,green,customer oriented services,customer oriented services,1
6,453160,DENILD680691144488472782_2,NaN,1515.565616,"['Retail_Non-Daily', 'Retail_Daily']",red,Retail_Non-Daily,Retail_Daily; Workers,0,Retail_Daily; Workers,Retail_Non-Daily,retail small scale,red,business oriented services,customer oriented services,0
7,21469,DENILD0100003euS,[Kunstatelier],93.639860,['Workers'],red,Leisure; Workers,(none),0,,Leisure; Workers,normal office,red,entertainment culture,(no class),0
